# Idiap / USSD teachers: 188 participant internal 60:20:20 protocol, seed 103

Run cells in order. This notebook starts with the **split and provenance audit**. It combines the 107 original TRAIN participants, 34 usable DEV participants (440 excluded), and 47 locally labeled original TEST participants into a new, stratified, participant-disjoint TRAIN-113 / VAL-37 / internal TEST-38 split. Internal TEST labels are used solely to stratify and freeze membership. Their features and predictions must remain unopened until the final evaluation.

**Checkpoint rule:** The published Idiap and USSD checkpoints already saw some participants who may now be in VAL or internal TEST. Do not reuse those weights for validation or final scoring. The requested transfer-learning experiments load the original checkpoints and fine-tune on TRAIN-113; their VAL and TEST results are exploratory because of prior participant exposure. This is an internal redistributed split, not an official AVEC test result. The provenance of local `full_test_split.csv` labels has not been independently verified. Earlier results on other splits cannot be compared as if this were an untouched test.

Stage 1 below saves only the split. We will add preprocessing/teacher stages to this same notebook after the split audit is confirmed. The older experiment folders are left untouched.

**Protocol revision:** This 60:20:20 split follows inspection of earlier 150/19/19 validation results. It retains all 19 previously closed test participants in the 38-person test, moves the previous 19-person validation group into training, and draws 19 more test plus 37 validation participants from the previous training group. The additional test participants appeared in earlier training runs. Treat this as a revised exploratory protocol; do not claim its test is pristine across all research iterations. Previous experiment outputs remain intact.


## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
if not (Path('/content/drive/MyDrive/DAIC_WOZ') / 'metadata').is_dir():
    drive.mount('/content/drive')
import base64, datetime, json, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Drive unavailable; reconnect Drive in Colab, then rerun: {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
CODE = Path('/content/reliability-aware-depression-kd-original188')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists(), f'Existing folder is not a Git worktree: {CODE}'


def safe_switch(target):
    """Switch this worktree, preserving dirty work before retrying a refused switch."""
    command = ['git', '-C', str(CODE), 'switch', '--detach', target]
    first = subprocess.run(command, text=True, capture_output=True)
    if first.returncode == 0:
        return
    status = subprocess.run(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'],
                            text=True, capture_output=True)
    if status.returncode != 0:
        raise RuntimeError(f'git switch failed:\n{first.stderr}\ngit status failed:\n{status.stderr}')
    if not status.stdout.strip():
        raise RuntimeError(f'git switch failed with a clean worktree. Git says:\n{first.stderr}')

    before = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                            text=True, capture_output=True)
    old_stash = before.stdout.strip() if before.returncode == 0 else ''
    stash_name = 'colab-preserved-before-switch-' + target[:12]
    saved = subprocess.run(['git', '-C', str(CODE), 'stash', 'push', '--include-untracked', '-m', stash_name],
                           text=True, capture_output=True)
    if saved.returncode != 0:
        raise RuntimeError(f'Git refused the switch and could not preserve local files.\n'
                           f'Switch error: {first.stderr}\nStash error: {saved.stderr}')
    current_stash = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                                   text=True, capture_output=True)
    stash_sha = current_stash.stdout.strip() if current_stash.returncode == 0 else ''
    if not stash_sha or stash_sha == old_stash:
        raise RuntimeError(f'Git refused the switch, and no new stash was created.\n'
                           f'Switch error: {first.stderr}\nStash output: {saved.stdout}{saved.stderr}')

    backup_root = DAIC / 'experiments' / 'colab_worktree_backups'
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup = backup_root / f'{stamp}_{target[:12]}'
    backup.mkdir(parents=True, exist_ok=False)
    patch = subprocess.run(['git', '-C', str(CODE), 'stash', 'show', '--patch', '--binary',
                            '--include-untracked', stash_sha], text=True, capture_output=True)
    if patch.returncode != 0 or not patch.stdout:
        raise RuntimeError(f'Local files remain safely in Git stash {stash_sha}, but Drive backup export failed. '
                           f'No switch was attempted. Details: {patch.stderr}')
    (backup / 'local_changes.patch').write_text(patch.stdout)
    (backup / 'git_status_before_switch.txt').write_text(status.stdout)
    (backup / 'manifest.json').write_text(json.dumps({
        'target_revision': target,
        'stash_commit': stash_sha,
        'stash_name': stash_name,
        'worktree': str(CODE),
        'original_switch_error': first.stderr,
        'backup_utc': stamp,
        'restore_note': 'Changes are retained in the worktree Git stash. Inspect the patch before applying it.'
    }, indent=2) + '\n')
    clean = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'], text=True)
    if clean.strip():
        raise RuntimeError(f'Files were stashed and backed up to {backup}, but the worktree is still dirty; '
                           f'no switch was attempted. Remaining status:\n{clean}')
    retry = subprocess.run(command, text=True, capture_output=True)
    if retry.returncode != 0:
        raise RuntimeError(f'Local work is preserved in stash {stash_sha} and backup {backup}. '
                           f'The switch still failed after cleanup:\n{retry.stderr}')
    print(f'Preserved local work in Git stash {stash_sha} and Drive backup {backup}')


if subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() != revision:
    safe_switch(revision)
assert subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
# Colab retains already imported modules from earlier revisions.
sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
sys.path.insert(0, str(CODE))
for name in list(sys.modules):
    if name == 'scripts' or name.startswith('scripts.'):
        del sys.modules[name]
assert (CODE / 'scripts/experiments/internal188/coverage.py').is_file(), 'Checkout is missing coverage.py'
print('Revision:', revision)


def refresh_code():
    """Update this existing worktree; no new /content code folder."""
    secret = userdata.get('GITHUB_TOKEN')
    assert secret, 'GITHUB_TOKEN missing from Colab Secrets'
    credentials = os.environ.copy()
    credentials.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
                       GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' +
                       base64.b64encode(f'x-access-token:{secret}'.encode()).decode())
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=credentials)
    latest = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    current = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
    if latest != current:
        safe_switch(latest)
    sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
    sys.path.insert(0, str(CODE))
    for name in list(sys.modules):
        if name == 'scripts' or name.startswith('scripts.'):
            del sys.modules[name]
    print('Code revision:', latest)
    del secret, credentials


## 2. Freeze the 113 / 37 / 38 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.internal188.split import main as split_stage
PRIOR_EXP = DAIC / 'experiments' / 'idiap_ussd_internal188_seed103'
EXP = DAIC / 'experiments' / 'idiap_ussd_internal188_60_20_20_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split'), '--prior-split-dir', str(PRIOR_EXP / 'split'), '--seed', '103'])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
assert 440 not in set(manifest.participant_id)
assert manifest.split.value_counts().to_dict() == {'train': 113, 'val': 37, 'student_test': 38}
prior = pd.read_csv(PRIOR_EXP / 'split' / 'manifest.csv')
assert set(prior.loc[prior.split.eq('student_test'), 'participant_id']) <= set(manifest.loc[manifest.split.eq('student_test'), 'participant_id'])
assert set(prior.loc[prior.split.eq('val'), 'participant_id']) <= set(manifest.loc[manifest.split.eq('train'), 'participant_id'])
print(pd.crosstab(manifest.split, manifest.label))
print('Source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('Saved:', EXP / 'split' / 'manifest.csv')


## 3. Audit TRAIN/VAL transcripts and ComParE16 cache

This reads audio feature arrays and transcript paths for **150 TRAIN/VAL participants only**. It does not open internal TEST recordings, transcripts or feature arrays. Existing feature caches may be shared; keep their directories available. If coverage is incomplete, stop and report the exact missing participant IDs before any training.


In [ ]:
from scripts.experiments.internal188.coverage import main as coverage_stage
cache_candidates = [DAIC / 'experiments' / 'ussd_compare16',
                    DAIC / 'experiments' / 'full188_seed42' / 'features',
                    DAIC / 'experiments' / 'full188_80_10_10_seed103' / 'features']
available = [p for p in cache_candidates if (p / 'participant_manifest.csv').is_file()]
print('Available feature caches:', [str(p) for p in available])
assert available, 'No ComParE16 feature manifest found'
coverage_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--output', str(EXP / 'coverage'),
                *[arg for p in available for arg in ('--feature-cache', str(p))]])
covered = pd.read_csv(EXP / 'coverage' / 'participant_manifest.csv')
assert covered.split.value_counts().to_dict() == {'train': 113, 'val': 37}
print('Coverage:', pd.crosstab(covered.split, covered.label))


## 4. Idiap checkpoint initialization and new fine-tuning

Run this after sections 1–3. It downloads the **exact published checkpoint and vectorizer**, verifies their Git blob hashes, builds a new TRAIN-113 graph and vocabulary, then transfers the output layer and first-layer columns for shared vocabulary tokens. Newly introduced tokens keep random initialization. It then fine-tunes a new teacher on TRAIN-113 and selects the checkpoint with VAL-37 macro F1. The published checkpoint already trained on some current VAL and internal TEST participants; this run is exploratory and cannot support a clean held-out test claim. Its output directory is separate from the fresh-weight experiment.


In [ ]:
refresh_code()
assert (CODE / 'scripts/experiments/internal188/idiap_source.py').is_file(), 'Stale CODE checkout: rerun notebook section 1 before training'
from scripts.experiments.internal188.idiap_source import ensure_source, ensure_published_weights
from scripts.experiments.internal188.train_text_teacher import main as text_teacher_stage
idiap_source = ensure_source(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'bias_in_daic-woz')
checkpoint, old_vectorizer = ensure_published_weights()
text_teacher_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                    '--coverage-dir', str(EXP / 'coverage'),
                    '--idiap-source', str(idiap_source),
                    '--published-checkpoint', str(checkpoint),
                    '--published-vectorizer', str(old_vectorizer),
                    '--output', str(EXP / 'teachers' / 'text_published_init'), '--seed', '103'])


In [ ]:
import json
pretrained_audit = json.loads((EXP / 'teachers' / 'text_published_init' / 'audit.json').read_text())
print('Transferred:', pretrained_audit['initialization'])
print('Prior exposure:', pretrained_audit['published_checkpoint_prior_exposure'])
print('Best epoch:', pretrained_audit['best_epoch'])
print('TRAIN:', {k: pretrained_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: pretrained_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## 5. Train USSD audio teacher with the published run #4 weights

This starts from the exact pinned `md_35_epochs.pth`, uses the **pinned normalization saved with that checkpoint**, and trains a new model using 8 sampled 384-frame segments per participant with a participant-level weighted loss. It prints TRAIN and VAL macro F1 each epoch. Epoch 0 is evaluated and remains eligible if fine-tuning degrades VAL macro F1. This adaptation objective is different from the author's original speaker-disentanglement objective. The original run #4 weights had prior exposure to 11 current VAL and 14 current internal TEST participants, so this result is exploratory. Internal TEST is never inferred.


In [ ]:
refresh_code()
from scripts.teachers.ussd_audio.common import _resolve_pinned_author_artifact, AUTHOR_RUN4_CHECKPOINT_SHA256
from scripts.teachers.ussd_audio import CHECKPOINT_NAME
from scripts.experiments.internal188.train_audio_teacher import main as audio_teacher_stage
ussd_checkpoint = _resolve_pinned_author_artifact(
    DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression',
    CHECKPOINT_NAME, AUTHOR_RUN4_CHECKPOINT_SHA256)
audio_teacher_stage(['--split-dir', str(EXP / 'split'), '--coverage-dir', str(EXP / 'coverage'),
                     '--features', str(EXP / 'coverage'), '--published-checkpoint', str(ussd_checkpoint),
                     '--author-root', str(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression'),
                     '--output', str(EXP / 'teachers' / 'audio_published_author_norm_v2'), '--seed', '103'])


In [ ]:
import json
audio_audit = json.loads((EXP / 'teachers' / 'audio_published_author_norm_v2' / 'audit.json').read_text())
print('Epoch 0 VAL:', {k: audio_audit['epoch_zero']['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('Audio best epoch:', audio_audit['best_epoch'])
print('Prior exposure:', audio_audit['published_checkpoint_prior_exposure'])
print('TRAIN:', {k: audio_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: audio_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## 6. Audit participant aggregation on the saved audio checkpoint

No further training. Score TRAIN and VAL segments from the selected published-initialized checkpoint using both mean segment probability and the author's hard segment majority vote. Confirm the recalculated mean probabilities match the saved teacher targets. The internal TEST is never opened.


In [ ]:
refresh_code()
from scripts.experiments.internal188.audit_audio_aggregation import main as audit_audio_aggregation
AUDIT = EXP / 'teachers' / 'audio_aggregation_audit_v2'
audit_audio_aggregation(['--split-dir', str(EXP / 'split'),
                         '--coverage-dir', str(EXP / 'coverage'),
                         '--teacher-dir', str(EXP / 'teachers' / 'audio_published_author_norm_v2'),
                         '--output', str(AUDIT)])


## 7. Clean audio experiment: fresh USSD architecture and balanced participant pairs

This is the independent audio teacher candidate for the 188-person split. The ComParE16 arrays remain label-independent, while normalization is fitted on TRAIN-113. No published DAIC-trained checkpoint or original author normalization is loaded. Every TRAIN batch samples one depressed and one non-depressed participant, with 8 segments each; loss is computed once per participant, without an extra class weight. Checkpoints are selected on VAL-37 macro F1. Internal TEST is closed. This is a new training objective using the USSD architecture, not a reproduction of the full author speaker-disentanglement loss.


In [ ]:
refresh_code()
from scripts.experiments.internal188.train_audio_teacher import main as audio_teacher_stage
CLEAN_AUDIO = EXP / 'teachers' / 'audio_fresh_balanced_v1'
audio_teacher_stage(['--split-dir', str(EXP / 'split'), '--coverage-dir', str(EXP / 'coverage'),
                     '--features', str(EXP / 'coverage'), '--output', str(CLEAN_AUDIO),
                     '--seed', '103', '--balanced-pairs', '--lr', '0.001'])


In [ ]:
import json
clean_audio = json.loads((CLEAN_AUDIO / 'audit.json').read_text())
print('Best epoch:', clean_audio['best_epoch'])
print('TRAIN:', {k: clean_audio['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: clean_audio['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('Provenance:', clean_audio['normalization_source'], clean_audio['published_checkpoint_prior_exposure'])


## 8A. Cache externally pretrained WavLM Base+ audio embeddings

This is the next audio teacher candidate. WavLM Base+ is pretrained outside DAIC-WOZ; **its weights stay frozen**. For each TRAIN or VAL participant, recreate participant-only speech using the original USSD transcript interval procedure, resample to 16 kHz, select eight evenly spaced 6-second crops, and store the mean and standard deviation of WavLM embeddings. The exact model revision and input hashes are recorded. Completed participant caches are reused after interruption. No internal TEST audio is opened. The first pass downloads the model and can take considerable time.


In [ ]:
refresh_code()
import os
os.environ['HF_HOME'] = str(EXP / 'models' / 'huggingface_cache')
from scripts.experiments.internal188.wavlm_features import main as wavlm_features_stage
WAVLM = EXP / 'features' / 'wavlm_base_plus_v1'
wavlm_features_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                      '--coverage-dir', str(EXP / 'coverage'), '--output', str(WAVLM),
                      '--batch-size', '2',
                      '--reuse-from', str(PRIOR_EXP / 'features' / 'wavlm_base_plus_v1')])


## 8B. Fit a participant-level teacher on TRAIN-113 only

A predeclared balanced logistic classifier uses the cached 1,536-dimensional participant vectors. Its scaler and classifier are fitted only on TRAIN-113. VAL-37 is reported at the fixed 0.5 threshold, without threshold or hyperparameter search. It exports TRAIN in-sample KD logits/probabilities. Keep internal TEST closed.


In [ ]:
refresh_code()
from scripts.experiments.internal188.train_wavlm_teacher import main as wavlm_teacher_stage
WAVLM_TEACHER = EXP / 'teachers' / 'wavlm_base_plus_v1'
wavlm_teacher_stage(['--split-dir', str(EXP / 'split'),
                     '--coverage-dir', str(EXP / 'coverage'),
                     '--embeddings', str(WAVLM), '--output', str(WAVLM_TEACHER),
                     '--seed', '103'])


## Stop after WavLM validation

Share TRAIN and VAL metrics and confusion matrices. The internal test remains closed. We will only consider WavLM KD targets if validation shows independent signal.


## 9. Audit the public Step-Audio2 DAIC-WOZ LoRA adapter (exploratory)

This candidate is already trained on DAIC-WOZ. Download its small metadata first and record the exact revision and base-model declaration. The adapter publisher has not supplied the participant-level training roster, so we cannot establish that it did not see the current VAL or internal TEST participants. The original canonical TRAIN roster alone overlaps 23 current VAL and 23 current internal TEST participants. **Do not include this checkpoint in the main clean teacher/KD comparison or open internal TEST.** This stage does not run inference or alter the existing teachers.

Set `--download-weights` only after checking the metadata and resources. Its Step-Audio2 base model must be acquired separately; a LoRA adapter alone cannot make predictions.


In [ ]:
refresh_code()
from scripts.experiments.internal188.stepaudio2_adapter_audit import main as step_audio_audit
STEP_AUDIO = EXP / 'teachers' / 'stepaudio2_external_adapter_audit'
step_audio_audit(['--split-dir', str(EXP / 'split'), '--output', str(STEP_AUDIO)])
print((STEP_AUDIO / 'audit.json').read_text())


## 10. Step-Audio2 pilot on T4 or larger GPU

The T4 uses BNB INT8 for the language model and FP16 for the unquantized audio path. A repository plugin casts only the audio-encoder input; it preserves the model's `forward` signature and the released LoRA weights. The one-person VAL pilot streams and saves its full log. On a larger GPU it uses the original unquantized BF16 path. Internal TEST stays closed.


In [ ]:
refresh_code()
import json, os, shutil, subprocess, sys
from scripts.experiments.internal188.prepare_stepaudio2_val import main as prepare_stepaudio2_val
from scripts.experiments.internal188.run_logged import run as run_logged
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert gpu_name, 'A GPU runtime is required'
if shutil.which('swift') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ms-swift'])
assert shutil.which('swift'), 'ms-swift installation did not provide swift CLI'
STEP_PILOT = EXP / 'teachers' / 'stepaudio2_exploratory_val_pilot'
pilot_command = prepare_stepaudio2_val(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--coverage-dir', str(EXP / 'coverage'), '--adapter-audit', str(STEP_AUDIO / 'audit.json'),
    '--output', str(STEP_PILOT), '--pilot-only'])
if 'T4' in gpu_name:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])
    plugin = CODE / 'scripts/experiments/internal188/stepaudio2_t4_fp16_plugin.py'
    assert plugin.is_file(), f'Missing T4 plugin: {plugin}'
    pilot_command += ['--quant_method', 'bnb', '--quant_bits', '8',
                      '--torch_dtype', 'float16', '--external_plugins', str(plugin)]
else:
    pilot_command += ['--torch_dtype', 'bfloat16']
print('GPU:', gpu_name, 'T4 INT8 pilot:', 'T4' in gpu_name)
pilot_result = STEP_PILOT / 'pilot_predictions.jsonl'
if pilot_result.is_file() and pilot_result.stat().st_size == 0: pilot_result.unlink()
if not pilot_result.is_file():
    env = os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    run_logged(pilot_command, STEP_PILOT / 'swift_infer.log', cwd=CODE, env=env)
assert pilot_result.is_file(), f'Inference produced no result: {pilot_result}'
pilot_rows = [json.loads(line) for line in pilot_result.read_text().splitlines() if line.strip()]
assert len(pilot_rows) == 1, f'Expected one pilot prediction, got {len(pilot_rows)}'
print('Raw pilot response:', pilot_rows[0])


## 10B. Exploratory Step-Audio2 validation on 37 participants

The full participant WAV exceeded T4 memory in the audio encoder, so participant speech is sliced into contiguous, non-overlapping 20-second windows. The prior full inference completed 1,140 windows, but some responses used a `<think>` block and stopped before a Yes/No label. This rerun first probes the previously incomplete VAL window with an explicit English-answer prefix. If it produces a direct Yes/No, the notebook runs all windows with the same prefix and an 8-token limit, then aggregates duration-weighted votes per participant. It reports participant-level F1 and a confusion matrix only. This remains exploratory: windowing and prompting may differ from the adapter author’s evaluation, and the adapter may have seen some VAL participants. Internal TEST remains unopened.


In [ ]:
refresh_code()
from datetime import datetime, timezone
import json, pandas as pd, re
from scripts.experiments.internal188.prepare_stepaudio2_val import main as prepare_stepaudio2_val
from scripts.experiments.internal188.evaluate_stepaudio2_val import main as evaluate_stepaudio2_val
STEP_VAL = EXP / 'teachers' / 'stepaudio2_exploratory_val37_chunked20s'
val_command = prepare_stepaudio2_val(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--coverage-dir', str(EXP / 'coverage'), '--adapter-audit', str(STEP_AUDIO / 'audit.json'),
    '--output', str(STEP_VAL), '--chunk-seconds', '20'])
if 'T4' in gpu_name:
    val_command += ['--quant_method', 'bnb', '--quant_bits', '8',
                    '--torch_dtype', 'float16', '--external_plugins', str(plugin)]
else:
    val_command += ['--torch_dtype', 'bfloat16']
# Probe the exact window that previously ended inside a think block.
rows = [json.loads(line) for line in (STEP_VAL / 'val_chunks_20s.jsonl').read_text().splitlines() if line.strip()]
probe = next(row for row in rows if Path(row['audios'][0]).name == '364_chunk_0068.wav')
probe_dir = STEP_VAL / 'answer_prefix_probe'; probe_dir.mkdir(exist_ok=True)
probe_jsonl = probe_dir / 'probe.jsonl'; probe_jsonl.write_text(json.dumps(probe) + '\n')
probe_result = probe_dir / 'probe_predictions.jsonl'
probe_command = val_command.copy()
probe_command[probe_command.index('--val_dataset') + 1] = str(probe_jsonl)
probe_command[probe_command.index('--result_path') + 1] = str(probe_result)
if not probe_result.is_file():
    env = os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    run_logged(probe_command, probe_dir / 'probe_infer.log', cwd=CODE, env=env)
probe_rows = [json.loads(line) for line in probe_result.read_text().splitlines() if line.strip()]
if len(probe_rows) != 1 or not re.match(r'^\s*(?:<英语>\s*)?(yes|no)\b',
                                        probe_rows[0].get('response') or '', flags=re.IGNORECASE):
    raise ValueError('Answer-prefix probe did not return a direct Yes/No. Full VAL inference was not started.')
print('Answer-prefix probe passed; starting consistent full VAL inference.')
val_result = STEP_VAL / 'val_chunks_20s_direct_predictions.jsonl'
expected_windows = len(pd.read_csv(STEP_VAL / 'val_chunks_20s_index.csv'))
if val_result.is_file() and sum(bool(line.strip()) for line in val_result.open()) != expected_windows:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    val_result.rename(STEP_VAL / f'val_chunks_20s_direct_predictions.partial.{stamp}.jsonl')
if not val_result.is_file():
    env = os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    run_logged(val_command, STEP_VAL / 'swift_infer_direct.log', cwd=CODE, env=env)
evaluate_stepaudio2_val(['--split-dir', str(EXP / 'split'),
    '--index', str(STEP_VAL / 'val_chunks_20s_index.csv'), '--generations', str(val_result),
    '--output', str(STEP_VAL / 'evaluation_direct')])


## 11. Continue adapter fine-tuning (exploratory, larger GPU)

Only run after reviewing all 37 exploratory VAL results. Unquantized Step-Audio2 training requires more GPU memory than the T4 pilot; this cell blocks T4. The released adapter's unknown prior DAIC-WOZ exposure remains, so any metrics are exploratory. Internal TEST stays closed.


In [ ]:
refresh_code()
from scripts.experiments.internal188.prepare_stepaudio2_sft import main as prepare_stepaudio2_sft
assert (STEP_VAL / 'evaluation' / 'val_metrics.json').is_file(), 'Run and review section 10B first'
assert 'T4' not in gpu_name, 'Section 11 unquantized SFT needs a larger GPU; T4 was used only for the INT8 pilot'
STEP_SFT = EXP / 'teachers' / 'stepaudio2_published_adapter_sft_exploratory'
sft_command = prepare_stepaudio2_sft(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--coverage-dir', str(EXP / 'coverage'), '--adapter-audit', str(STEP_AUDIO / 'audit.json'),
    '--output', str(STEP_SFT)])
run_logged(sft_command, STEP_SFT / 'swift_sft.log', cwd=CODE)
